# 🎥 Advanced Video Analysis System

## Complete Implementation:
- ✅ **Object Detection Results** - Detailed metrics and visualizations
- ✅ **Multi-Object Tracking Results** - Track persistence analysis
- ✅ **Trajectory Prediction** - Future position forecasting
- ✅ **Depth Estimation** - MiDaS monocular depth
- ✅ **Collision Risk Evaluation** - Real-time safety assessment

---

## 📦 Step 1: Install All Dependencies

In [2]:
# Core ML/CV libraries
!pip install -q ultralytics opencv-python-headless deep-sort-realtime

# Deep learning frameworks
!pip install -q torch torchvision timm

# Data analysis and visualization
!pip install -q matplotlib seaborn pandas scipy

print('\n✅ All dependencies installed successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 70.0 MB/s eta 0:00:00

✅ All dependencies installed successfully!


## 📚 Step 2: Import Libraries

In [3]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from collections import defaultdict, deque
import math
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
import os
import json

# Configure visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

print('✅ All libraries imported!')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All libraries imported!


## ⚙️ Step 3: Configuration

In [4]:
# Model settings
MODEL_NAME = '/content/best.pt'
CONF_THRESHOLD = 0.5

# Tracking
MAX_AGE = 30
N_INIT = 3
TRAIL_LENGTH = 30

# Trajectory prediction
PREDICTION_FRAMES = 15
HISTORY_SIZE = 10

# Collision detection
COLLISION_DIST = 100  # pixels
TTC_THRESHOLD = 2.0   # seconds

# Colors (BGR)
COLOR_BOX = (0, 255, 0)
COLOR_VEL = (0, 0, 255)
COLOR_PRED = (255, 255, 0)
COLOR_TRAIL = (255, 0, 255)

print('✅ Configuration set!')

✅ Configuration set!


## 🤖 Step 4: Load Models

In [5]:
# Load YOLO
print(f'Loading {MODEL_NAME}...')
yolo_model = YOLO(MODEL_NAME)
print(f'✅ YOLO loaded ({len(yolo_model.names)} classes)')

# Initialize tracker
tracker = DeepSort(max_age=MAX_AGE, n_init=N_INIT)
print('✅ Tracker initialized!')

Loading /content/best.pt...
✅ YOLO loaded (27 classes)
✅ Tracker initialized!


**Train**

In [ ]:
# ========================================
# 📦 SETUP DATASET AND TRAIN YOLO MODEL
# ========================================
from google.colab import drive
from ultralytics import YOLO
import zipfile
import os
import yaml

# 1. Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Setup paths
ZIP_PATH = '/content/drive/MyDrive/499_Dataset_Collection/BRSDD_Shrinked.zip'  # ⬅️ CHANGE THIS
DATASET_PATH = '/content/dataset'

# 3. Extract dataset
print(f"\n📦 Extracting dataset...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATASET_PATH)
print(f"✅ Extracted to: {DATASET_PATH}")

# 4. Find data.yaml
DATA_YAML_PATH = os.path.join(DATASET_PATH, 'data.yaml')
if not os.path.exists(DATA_YAML_PATH):
    for root, dirs, files in os.walk(DATASET_PATH):
        if 'data.yaml' in files:
            DATA_YAML_PATH = os.path.join(root, 'data.yaml')
            DATASET_PATH = root
            break

print(f"\n✅ Found data.yaml at: {DATA_YAML_PATH}")

# 5. Update data.yaml paths to point to Colab location
with open(DATA_YAML_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the path in data.yaml
data_config['path'] = DATASET_PATH
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_config, f)

print(f"\n📄 Updated data.yaml:")
print(f"  path: {DATASET_PATH}")
print(f"  train: {data_config.get('train', 'N/A')}")
print(f"  val: {data_config.get('val', 'N/A')}")
print(f"  nc: {data_config.get('nc', 'N/A')}")

# 6. Verify dataset structure
print("\n📂 Dataset structure:")
for folder in ['train', 'valid', 'test']:
    folder_path = os.path.join(DATASET_PATH, folder, 'images')
    if os.path.exists(folder_path):
        num_images = len([f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  ✅ {folder}: {num_images} images")

# 7. Train the model
print("\n🚀 Starting training...")
model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    name='my_custom_model',
    patience=50,
    device=0,
    project='/content/drive/MyDrive/499_Dataset_Collection',
)

print("\n✅ Training complete!")
print(f"📊 Results: /content/drive/MyDrive/499_Dataset_Collection")

📁 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📦 Extracting dataset...
✅ Extracted to: /content/dataset

✅ Found data.yaml at: /content/dataset/BRSDD_Shrinked/data.yaml

📄 Updated data.yaml:
  path: /content/dataset/BRSDD_Shrinked
  train: ../train/images
  val: ../valid/images
  nc: 19

📂 Dataset structure:
  ✅ train: 5090 images
  ✅ valid: 740 images
  ✅ test: 568 images

🚀 Starting training...
Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/BRSDD_Shrinked/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynami

In [ ]:
# Resume the model
print("\n🚀 Resume trained model...")
model = YOLO('/content/runs/detect/runs/detect/my_custom_model2/weights/last.pt')
# resume if epoch left to train
# model.train(resume=True)


🚀 Resume trained model...


## 🛠️ Step 6: Utility Functions

In [6]:
def calc_velocity(curr, prev, fps=30):
    if prev is None:
        return 0, 0, 0, 0
    vx = (curr[0] - prev[0]) * fps
    vy = (curr[1] - prev[1]) * fps
    speed = math.sqrt(vx**2 + vy**2)
    angle = math.degrees(math.atan2(vy, vx))
    return vx, vy, speed, angle

def get_direction(angle):
    """Returns ASCII direction labels instead of Unicode arrows"""
    # Normalize angle to 0-360
    angle = angle % 360

    # 8 cardinal directions
    if 337.5 <= angle or angle < 22.5:
        return 'E'   # East (right)
    elif 22.5 <= angle < 67.5:
        return 'SE'  # Southeast
    elif 67.5 <= angle < 112.5:
        return 'S'   # South (down)
    elif 112.5 <= angle < 157.5:
        return 'SW'  # Southwest
    elif 157.5 <= angle < 202.5:
        return 'W'   # West (left)
    elif 202.5 <= angle < 247.5:
        return 'NW'  # Northwest
    elif 247.5 <= angle < 292.5:
        return 'N'   # North (up)
    else:
        return 'NE'  # Northeast

def predict_trajectory(history, n=15):
    if len(history) < 2:
        return []
    recent = list(history)[-min(5, len(history)):]
    vx = np.mean([recent[i][0]-recent[i-1][0] for i in range(1, len(recent))])
    vy = np.mean([recent[i][1]-recent[i-1][1] for i in range(1, len(recent))])
    last = recent[-1]
    return [(int(last[0]+vx*i), int(last[1]+vy*i)) for i in range(1, n+1)]

def calc_distance(H, f, N_p, h_p, S):
    """
    Parameters:
    - H : Real object height (same units as distance you want)
    - f : Focal length (same units as sensor height, e.g., mm)
    - N_p : Image height in pixels
    - h_p : Object height in image (pixels)
    - S : Sensor height (same units as focal length, e.g., mm)
    """
    if h_p == 0 or S == 0:
        return float('inf')  # Avoid division by zero
    D = (H * f * N_p) / (h_p * S)
    return D

def calc_camera_ttc(obj_pos, obj_vel, cam_pos=(0, 0), eps=1e-6):
    """
    Time-to-collision between object and camera (ego)

    obj_pos : (x, y) pixel position of object
    obj_vel : (vx, vy) velocity in px/sec
    cam_pos : camera reference point (default overwritten per-frame)
    """
    rx = obj_pos[0] - cam_pos[0]
    ry = obj_pos[1] - cam_pos[1]

    rvx = obj_vel[0]
    rvy = obj_vel[1]

    v2 = rvx**2 + rvy**2
    if v2 < eps:
        return None

    ttc = - (rx * rvx + ry * rvy) / v2
    return ttc if ttc > 0 else None

print('✅ Utility functions defined!')

✅ Utility functions defined!


## 🎬 Step 7: Main Processing Function

In [7]:
def process_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print('❌ Cannot open video')
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f'📹 {w}x{h} @ {fps}fps, {total} frames')

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    track_history = defaultdict(lambda: deque(maxlen=HISTORY_SIZE))
    track_pos = {}
    track_vel = {}

    frame_num = 0
    print('\n🔄 Processing...\n')

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_num += 1

        # Detect
        results = yolo_model(frame, conf=CONF_THRESHOLD, verbose=False)[0]
        detections = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            cls = yolo_model.names[int(box.cls[0])]
            detections.append(([x1, y1, x2-x1, y2-y1], conf, cls))

        # Track
        tracks = tracker.update_tracks(detections, frame=frame)
        current_tracks = {}
        active = 0

        for track in tracks:
            if not track.is_confirmed():
                continue

            active += 1
            tid = track.track_id
            x1, y1, x2, y2 = map(int, track.to_ltrb())
            center = ((x1+x2)//2, (y1+y2)//2)
            cls = track.get_det_class()
            conf = track.get_det_conf()

            # Velocity
            prev = track_pos.get(tid)
            vx, vy, speed, angle = calc_velocity(center, prev, fps)
            direction = get_direction(angle)

            track_pos[tid] = center
            track_vel[tid] = (vx, vy)
            track_history[tid].append(center)

            # Real-world distance (in meters, for example)
            box_height_px = y2 - y1
            obj_distance = calc_distance(H= 1.65,  # REAL_HEIGHTS.get(cls, 1.5) default 1.5m
                                         f= 5.37, # FOCAL_LENGTH
                                         N_p=h, #
                                         h_p=box_height_px,
                                         S=5.32) #SENSOR_HEIGHT


            current_tracks[tid] = {'pos': center, 'vel': (vx, vy), 'cls': cls, 'box': (x1, y1, x2, y2), 'dist': obj_distance}

            # Predict trajectory
            preds = predict_trajectory(track_history[tid], PREDICTION_FRAMES)

            # Draw bbox
            cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_BOX, 2)
            label = f'ID:{tid} {cls}'
            if conf is not None:
                label += f' {conf:.2f}'
            label += f' D:{obj_distance:.1f}m'
            cv2.putText(frame, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)

            # Velocity arrow and direction text - FIXED
            if speed > 5:
                end = (int(center[0]+vx/3), int(center[1]+vy/3))
                cv2.arrowedLine(frame, center, end, COLOR_VEL, 2, tipLength=0.3)
                # Use ASCII text instead of Unicode arrows
                cv2.putText(frame, f'{direction} {speed:.0f}px/s', (x1, y2+20),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_VEL, 2)

            # Trail
            if len(track_history[tid]) > 1:
                hist = list(track_history[tid])
                for i in range(1, len(hist)):
                    cv2.line(frame, hist[i-1], hist[i], COLOR_TRAIL, 2)

            # Predicted path
            if len(preds) > 1:
                for i in range(len(preds)-1):
                    cv2.line(frame, preds[i], preds[i+1], COLOR_PRED, 2, cv2.LINE_AA)


        # ---------------- RISK ANALYSIS ----------------
        risks = []

        # Camera assumed at bottom-center of frame
        cam_pos = (w // 2, h)

        for tid, data in current_tracks.items():
            pos = data['pos']
            vel = data['vel']

            ttc = calc_camera_ttc(pos, vel, cam_pos)

            # Pixel distance to camera
            dist = math.sqrt((pos[0] - cam_pos[0])**2 + (pos[1] - cam_pos[1])**2)

            level = None
            if ttc and ttc < TTC_THRESHOLD:
                level = 'HIGH' if ttc < 0.5 else ('MEDIUM' if ttc < 1.0 else 'LOW')
            elif dist < COLLISION_DIST:
                level = 'MEDIUM'

            if level:
                risks.append({'id': tid, 'ttc': ttc, 'dist': dist, 'level': level})

                # Visual warning
                color = {'HIGH': (0,0,255), 'MEDIUM': (0,165,255), 'LOW': (0,255,255)}[level]
                cv2.line(frame, cam_pos, pos, color, 2)
                mid = ((cam_pos[0] + pos[0]) // 2, (cam_pos[1] + pos[1]) // 2)
                txt = f"{level} RISK"
                if ttc:
                    txt += f" ({ttc:.1f}s)"
                cv2.putText(frame, txt, mid, cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # NEW: Display TTC value on each tracked object (even if not high risk)
            box = data['box']
            x1, y1, x2, y2 = box
            if ttc is not None:
                ttc_text = f"TTC:{ttc:.1f}s"
                ttc_color = (0, 255, 0) if ttc > 2.0 else ((0, 165, 255) if ttc > 1.0 else (0, 0, 255))
                cv2.putText(frame, ttc_text, (x1, y2+40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, ttc_color, 2)


        # Info overlay
        cv2.putText(frame, f'Frame: {frame_num}/{total}', (10,30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
        cv2.putText(frame, f'Objects: {active}', (10,55),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
        if risks:
            cv2.putText(frame, f'⚠ Risks: {len(risks)}', (10,80),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

        out.write(frame)

        if frame_num % 30 == 0:
            print(f'Progress: {frame_num/total*100:.1f}% | Objects: {active} | Risks: {len(risks)}')

    cap.release()
    out.release()

    print(f'\n✅ Processing complete!')
    print(f'📁 Output: {output_path}')

print('✅ Processing function defined!')

✅ Processing function defined!


## 📤 Step 8: Upload Video

In [13]:
input_video = '/content/drive/MyDrive/499_Dataset_Collection/Dhaka - comilla com1.mp4'
print(f'✅ Using video from Drive: {input_video}')

# print('📤 Select video file...')
# uploaded = files.upload()
# input_video = list(uploaded.keys())[0]
# print(f'✅ Uploaded: {input_video}')

✅ Using video from Drive: /content/drive/MyDrive/499_Dataset_Collection/Dhaka - comilla com1.mp4


## 🚀 Step 9: Run Processing

In [ ]:
output_video = 'output_tracked.mp4'
process_video(input_video, output_video)

📹 1280x720 @ 59fps, 18784 frames

🔄 Processing...

Progress: 0.2% | Objects: 2 | Risks: 0
Progress: 0.3% | Objects: 3 | Risks: 0
Progress: 0.5% | Objects: 4 | Risks: 0
Progress: 0.6% | Objects: 5 | Risks: 1
Progress: 0.8% | Objects: 3 | Risks: 0
Progress: 1.0% | Objects: 3 | Risks: 0
Progress: 1.1% | Objects: 4 | Risks: 0
Progress: 1.3% | Objects: 2 | Risks: 0
Progress: 1.4% | Objects: 4 | Risks: 0
Progress: 1.6% | Objects: 1 | Risks: 0
Progress: 1.8% | Objects: 1 | Risks: 0
Progress: 1.9% | Objects: 1 | Risks: 1
Progress: 2.1% | Objects: 1 | Risks: 1
Progress: 2.2% | Objects: 2 | Risks: 1
Progress: 2.4% | Objects: 1 | Risks: 1
Progress: 2.6% | Objects: 1 | Risks: 0
Progress: 2.7% | Objects: 2 | Risks: 0
Progress: 2.9% | Objects: 4 | Risks: 0
Progress: 3.0% | Objects: 2 | Risks: 0
Progress: 3.2% | Objects: 2 | Risks: 2
Progress: 3.4% | Objects: 3 | Risks: 0
Progress: 3.5% | Objects: 1 | Risks: 0
Progress: 3.7% | Objects: 2 | Risks: 0
Progress: 3.8% | Objects: 4 | Risks: 0
Progress: 4.0

## 💾 Step 16: Download All Results

In [10]:
print('💾 Downloading...\n')

# Videos
if os.path.exists(output_video):
    files.download(output_video)

print('\n✅ Downloads complete!')

💾 Downloading...


✅ Downloads complete!


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!yolo export model=yolov8n.pt format=tflite int8

Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
E0000 00:00:1769632195.049104    3552 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769632195.054222    3552 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769632195.067062    3552 computation_placer.cc:177] computation placer already registered. Please check 